# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)

This notebook simulates OMOP transformation and probabilistic patient linkage using Splink.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
import splink.spark.comparison_library as cl
import splink.spark.comparison_template_library as ctl
from splink.spark.linker import SparkLinker

spark = SparkSession.builder \
    .appName("Day05_OMOP_Linkage") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "1") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir("/tmp/splink_checkpoints")

print("Spark session initialized.")

In [ ]:
# 1. OMOP Transformation Mock
data = [("P001", "J45.909", "2023-01-15"), ("P002", "E11.9", "2023-02-20")]
columns = ["patient_id_fpe", "diagnosis_code", "service_date"]
input_df = spark.createDataFrame(data, columns)

vocab_data = [("J45.909", 317009), ("E11.9", 201820)]
vocab_columns = ["concept_code", "concept_id"]
vocab_df = spark.createDataFrame(vocab_data, vocab_columns)

condition_df = input_df.join(vocab_df, input_df.diagnosis_code == vocab_df.concept_code, "left") \
    .select(
        col("patient_id_fpe").alias("person_id"),
        col("concept_id").alias("condition_concept_id"),
        col("service_date").alias("condition_start_date"),
        lit(32020).alias("condition_type_concept_id"),
        col("diagnosis_code").alias("condition_source_value"),
        current_timestamp().alias("etl_timestamp")
    )

print("OMOP condition_occurrence:")
condition_df.show()

In [ ]:
# 2. Splink Record Linkage Mock
linker_data = [
    ("1", "John", "Doe", "1980-01-01", "A"),
    ("2", "Jon", "Doe", "1980-01-01", "B"),
    ("3", "Jane", "Smith", "1990-05-15", "A")
]
linker_df = spark.createDataFrame(linker_data, ["unique_id", "first_name", "last_name", "dob", "source_dataset"])

settings = {
    "link_type": "dedupe_only",
    "blocking_rules_to_generate_predictions": [
        "l.last_name = r.last_name"
    ],
    "comparisons": [
        cl.exact_match("first_name"),
        cl.exact_match("last_name"),
        cl.exact_match("dob")
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

linker = SparkLinker(linker_df, settings, spark=spark)
linker.estimate_u_using_random_sampling(max_pairs=1e6)

df_predict = linker.predict(threshold_match_probability=0.5)

print("Splink Linkage Predictions:")
df_predict.as_pandas_dataframe()[["first_name_l", "first_name_r", "match_probability"]]